### DATA TRANSFORMATION

In [16]:
import pandas as pd
import numpy as np
import re
from datetime import date, datetime

## All strata sheet

In [58]:
# define the path and sheet
path = "dataset/DATA_2017 copy.xls"   # your file
sheet = "ad"                   # your sheet name

# 1) Read without header, find the row that contains "Code", then re-read with that row as header
raw = pd.read_excel(path, sheet_name=sheet, header=None)
hdr = raw.index[raw.apply(lambda r: r.astype(str).str.strip().eq("Code")).any(axis=1)][0]
df  = pd.read_excel(path, sheet_name=sheet, header=hdr)

# 2) Make a ring_type column from banner rows and drop banners
name_col = "Area/Ring/SA"  # second column in your screenshot
# match exact the column name
exact_area = df[name_col].astype(str)\
             .str.replace(r"\s+", " ", regex=True)\
             .str.strip()
banner_names = {"Greater Sydney", "Inner Ring", "Middle Ring", "Outer Ring", "Rest of GMR", "GMR", 'Rest of New South Wales', 'NEW SOUTH WALES'}
banner_mask  = exact_area.isin(banner_names)
# create ring type variable
df["ring_type"] = df[name_col].where(df[name_col]\
                                     .isin({'Inner Ring', 'Middle Ring', 'Outer Ring', 'Rest of GMR'}))
df["ring_type"] = df["ring_type"].ffill()
# df = df.loc[~banner_mask].copy()

# 3) Identify the month columns
def is_month_col(c):
    if isinstance(c, (pd.Timestamp, datetime, date)):
        return True
    s = str(c)
    return bool(re.fullmatch(r"[A-Za-z]{3}-\d{2}", s))

month_cols = [c for c in df.columns if is_month_col(c)]

# 4) Clean values
df[month_cols] = (df[month_cols]
                  .replace({"s": np.nan, "-": np.nan, "": np.nan})
                  .apply(pd.to_numeric, errors="coerce"))

# 5) Wide → long
long_df = df.melt(
    id_vars=["Code", name_col, "ring_type"],
    value_vars=month_cols,
    var_name="period",
    value_name="median_price"
)

# 6) Time features
long_df["date"]    = pd.to_datetime(long_df["period"], format="%b-%y")
long_df["year"]    = long_df["date"].dt.year
long_df["month"]   = long_df["date"].dt.month
long_df["quarter"] = long_df["date"].dt.to_period("Q").astype(str)

# 7) Final output
output = (long_df[["Code","ring_type",name_col,"date","year","month","quarter","median_price"]]
       .rename(columns={
           "Code":"code", 
           name_col:"area"})
       .sort_values(["ring_type","area","date"])
       .reset_index(drop=True))


/var/folders/fs/vpv7t9012wxdx__nwt82lmpm0000gn/T/ipykernel_31370/3117340761.py:35: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"s": np.nan, "-": np.nan, "": np.nan})


In [59]:
# check again for the data for 2017
output['year'].value_counts()
output_2017only = output[output['year'] == 2016]
output_2017only

,code,ring_type,area,date,year,month,quarter,median_price
100,150,Inner Ring,Ashfield,2016-03-01,2016,3,2016Q1,740.0
101,150,Inner Ring,Ashfield,2016-06-01,2016,6,2016Q2,854.0
102,150,Inner Ring,Ashfield,2016-09-01,2016,9,2016Q3,724.0
103,150,Inner Ring,Ashfield,2016-12-01,2016,12,2016Q4,800.0
205,1100,Inner Ring,Botany Bay,2016-03-01,2016,3,2016Q1,785.0
...,...,...,...,...,...,...,...,...
6088,8450,Rest of GMR,Wollongong,2016-12-01,2016,12,2016Q4,635.0
6190,0,NaN,Greater Sydney,2016-03-01,2016,3,2016Q1,760.0
6191,0,NaN,Greater Sydney,2016-06-01,2016,6,2016Q2,780.0
6192,0,NaN,Greater Sydney,2016-09-01,2016,9,2016Q3,788.0


In [60]:
special_col = {"Greater Sydney", "GMR", 'Rest of New South Wales', 'NEW SOUTH WALES'}
# output = output.where(output['area'] == special_col)
check = output.loc[output['area'].isin(special_col)]
# filling ring type for area "Greate Sydney" by itself
output['ring_type'] = output['ring_type'].fillna('GreaterSydney')

In [76]:
#adding strata-title
output['strata_title'] = "all dwelling"
output.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6195 entries, 0 to 6194
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   code          6195 non-null   int64         
 1   ring_type     6195 non-null   object        
 2   area          6195 non-null   object        
 3   date          6195 non-null   datetime64[ns]
 4   year          6195 non-null   int32         
 5   month         6195 non-null   int32         
 6   quarter       6195 non-null   object        
 7   median_price  6082 non-null   float64       
 8   strata_title  6195 non-null   object        
dtypes: datetime64[ns](1), float64(1), int32(2), int64(1), object(4)
memory usage: 387.3+ KB


Most of original data has been merged into a new data frame:
- 2017 has only one quarter beginning at March. 

## Non strata dwelling and Yes strata dwelling

In [7]:
# 1) Read without header, find the row that contains "Code", then re-read with that row as header
raw = pd.read_excel(path, sheet_name='ns', header=None)
hdr = raw.index[raw.apply(lambda r: r.astype(str).str.strip().eq("Area/Ring/SA")).any(axis=1)][0]
df  = pd.read_excel(path, sheet_name='ns', header=hdr)

In [62]:
# 2) Make a ring_type column from banner rows and drop banners
name_col = "Area/Ring/SA" 
# match exact the column name
exact_area = df[name_col].astype(str)\
                         .str.replace(r"\s+", " ", regex=True)\
                         .str.strip()
banner_names = {"Greater Sydney", "Inner Ring", "Middle Ring", "Outer Ring", "Rest of GMR", "GMR", 'Rest of New South Wales', 'NEW SOUTH WALES'}
banner_mask  = exact_area.isin(banner_names)
# create ring type variable
df["ring_type"] = df[name_col].where(df[name_col]\
                                     .isin({'Inner Ring', 'Middle Ring', 'Outer Ring', 'Rest of GMR'}))
df["ring_type"] = df["ring_type"].ffill()
# df = df.loc[~banner_mask].copy()

# 3) Identify the month columns
def is_month_col(c):
    if isinstance(c, (pd.Timestamp, datetime, date)):
        return True
    s = str(c)
    return bool(re.fullmatch(r"[A-Za-z]{3}-\d{2}", s))

month_cols = [c for c in df.columns if is_month_col(c)]

# 5) Wide → long
long_df = df.melt(
    id_vars=["Code", name_col, "ring_type"],
    value_vars=month_cols,
    var_name="period",
    value_name="median_price"
)

# 6) Time features
long_df["date"]    = pd.to_datetime(long_df["period"], format="%b-%y")
long_df["year"]    = long_df["date"].dt.year
long_df["month"]   = long_df["date"].dt.month
long_df["quarter"] = long_df["date"].dt.to_period("Q").astype(str)

# 7) Final output
output_ns = (long_df[["Code","ring_type",name_col,"date","year","month","quarter","median_price"]]
       .rename(columns={"Code":"code", name_col:"area"})
       .sort_values(["ring_type","area","date"])
       .reset_index(drop=True))

output_ns['strata_title'] = 'ns'

In [65]:
# looking at non-strata sheet
filter = output_ns['area'] == 'Greater Sydney'
output_ns[filter]

# filling ring_type for "Greater Sydney" by its own. name 
output_ns['ring_type'] = output_ns['ring_type'].fillna('Greater Sydney')

In [75]:
output_ns

,code,ring_type,area,date,year,month,quarter,median_price,strata_title
0,150,Inner Ring,Ashfield,1991-03-01,1991,3,1991Q1,137.0,ns
1,150,Inner Ring,Ashfield,1991-06-01,1991,6,1991Q2,141.0,ns
2,150,Inner Ring,Ashfield,1991-09-01,1991,9,1991Q3,142.0,ns
3,150,Inner Ring,Ashfield,1991-12-01,1991,12,1991Q4,145.0,ns
4,150,Inner Ring,Ashfield,1992-03-01,1992,3,1992Q1,NaN,ns
...,...,...,...,...,...,...,...,...,...
6190,0,Greater Sydney,Greater Sydney,2016-03-01,2016,3,2016Q1,760.0,ns
6191,0,Greater Sydney,Greater Sydney,2016-06-01,2016,6,2016Q2,780.0,ns
6192,0,Greater Sydney,Greater Sydney,2016-09-01,2016,9,2016Q3,788.0,ns
6193,0,Greater Sydney,Greater Sydney,2016-12-01,2016,12,2016Q4,840.0,ns


In [77]:
# yes strata
raw = pd.read_excel(path, sheet_name="s", header=None)
hdr = raw.index[raw.apply(lambda r: r.astype(str).str.strip().eq("Code")).any(axis=1)][0]
df  = pd.read_excel(path, sheet_name="s", header=hdr)

# 2) Make a ring_type column from banner rows and drop banners
name_col = "Area/Ring/SA"  # second column in your screenshot
# match exact the column name
exact_area = df[name_col].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
banner_names = {"Greater Sydney", "Inner Ring", "Middle Ring", "Outer Ring", "Rest of GMR", "GMR", 'Rest of New South Wales', 'NEW SOUTH WALES'}
banner_mask  = exact_area.isin(banner_names)
# create ring type variable
df["ring_type"] = df[name_col].where(df[name_col]\
                                     .isin({'Inner Ring', 'Middle Ring', 'Outer Ring', 'Rest of GMR'}))
df["ring_type"] = df["ring_type"].ffill()
# df = df.loc[~banner_mask].copy()

# 3) Identify the month columns
def is_month_col(c):
    if isinstance(c, (pd.Timestamp, datetime, date)):
        return True
    s = str(c)
    return bool(re.fullmatch(r"[A-Za-z]{3}-\d{2}", s))

month_cols = [c for c in df.columns if is_month_col(c)]
# Cleaning value 
df[month_cols] = (df[month_cols]
                  .replace({"s": np.nan, "-": np.nan, "": np.nan})
                  .apply(pd.to_numeric, errors="coerce"))

# 5) Wide → long
long_df = df.melt(
    id_vars=["Code", name_col, "ring_type"],
    value_vars=month_cols,
    var_name="period",
    value_name="median_price"
)

# 6) Time features
long_df["date"]    = pd.to_datetime(long_df["period"], format="%b-%y")
long_df["year"]    = long_df["date"].dt.year
long_df["month"]   = long_df["date"].dt.month
long_df["quarter"] = long_df["date"].dt.to_period("Q").astype(str)

# 7) Final output
output_strata = (long_df[["Code","ring_type",name_col,"date","year","month","quarter","median_price"]]
       .rename(columns={"Code":"code", name_col:"area"})
       .sort_values(["ring_type","area","date"])
       .reset_index(drop=True))

# define strata title
output_strata['strata title'] = 'strata'

# filling ring_type for "Greater Sydney" by its own. name 
output_strata['ring_type'] = output_strata['ring_type'].fillna('Greater Sydney')


/var/folders/fs/vpv7t9012wxdx__nwt82lmpm0000gn/T/ipykernel_31370/1298330333.py:28: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"s": np.nan, "-": np.nan, "": np.nan})


In [78]:
# looking at strata sheet
output_strata.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6195 entries, 0 to 6194
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   code          6195 non-null   int64         
 1   ring_type     6195 non-null   object        
 2   area          6195 non-null   object        
 3   date          6195 non-null   datetime64[ns]
 4   year          6195 non-null   int32         
 5   month         6195 non-null   int32         
 6   quarter       6195 non-null   object        
 7   median_price  5783 non-null   float64       
 8   strata title  6195 non-null   object        
dtypes: datetime64[ns](1), float64(1), int32(2), int64(1), object(4)
memory usage: 387.3+ KB


## Data cleaning and Analysis

In [81]:
# count distinct value for each ring type 
print('Distinct type of ring:',output['ring_type'].value_counts())


Distinct type of ring: ring_type
Outer Ring       1890
Middle Ring      1680
Inner Ring       1260
Rest of GMR      1260
GreaterSydney     105
Name: count, dtype: int64


In [13]:
# Replace NA with median for that year
output['median_price'] = output.groupby('year')['median_price']\
                       .transform(lambda x: x.fillna(x.median())) 
'''
Replace NA by median of that year instead of the median for the whole period
- Preserves yearly price levels and trends.
- Accounts for long-term inflation, market shifts, and local patterns.
'''


'\nReplace NA by median of that year instead of the median for the whole period\n- Preserves yearly price levels and trends.\n- Accounts for long-term inflation, market shifts, and local patterns.\n'

In [82]:
# def function to replace Na with median for that year in total 3 sheet "ad", 'ns', 's'
def replace_median(sheet):
    sheet['median_price'] = sheet.groupby('year')['median_price']\
        .transform(lambda x: x.fillna(x.median()))
    return sheet
# replace_median 3 sheet
output = replace_median(output)
output_ns = replace_median(output_ns)
output_strata = replace_median(output_strata)

In [89]:
# Get info about all dataset
def print_info(sheet):
    tittle = sheet['strata_title'].iloc[0] if 'strata_title' in sheet.columns else 'unknown'
    print(f'info about sheet {tittle}:')
    info = sheet.info()
    return info

print_info(output_strata) #alter for three sheet

info about sheet unknown:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6195 entries, 0 to 6194
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   code          6195 non-null   int64         
 1   ring_type     6195 non-null   object        
 2   area          6195 non-null   object        
 3   date          6195 non-null   datetime64[ns]
 4   year          6195 non-null   int32         
 5   month         6195 non-null   int32         
 6   quarter       6195 non-null   object        
 7   median_price  6195 non-null   float64       
 8   strata title  6195 non-null   object        
dtypes: datetime64[ns](1), float64(1), int32(2), int64(1), object(4)
memory usage: 387.3+ KB


## Adjusting price - REAL PRICE vs NOMINAL PRICE

- <code> value in year x = value in year Y * CPI(x)/ CPI(y) </code>

In [86]:
# loading the CPI index table 
cpi = pd.read_excel('dataset/CPI.xlsx', sheet_name= 'Cpi')
cpi.dtypes
print(cpi)

         year  month   year2    CPI
0  1991-06-01       6   1991   57.2
1  1992-06-01       6   1992   54.0
2  1993-06-01       6   1993   53.7
3  1994-06-01       6   1994   53.5
4  1995-06-01       6   1995   59.5
5  1996-06-01       6   1996   61.4
6  1997-06-01       6   1997   56.8
7  1998-06-01       6   1998   56.6
8  1999-06-01       6   1999   58.2
9  2000-06-01       6   2000   61.2
10 2001-06-01       6   2001   65.4
11 2002-06-01       6   2002   67.2
12 2003-06-01       6   2003   68.9
13 2004-06-01       6   2004   71.1
14 2005-06-01       6   2005   73.7
15 2006-06-01       6   2006   75.6
16 2007-06-01       6   2007   77.9
17 2008-06-01       6   2008   82.4
18 2009-06-01       6   2009   86.9
19 2010-06-01       6   2010   92.1
20 2011-06-01       6   2011   96.8
21 2012-06-01       6   2012  101.0
22 2013-06-01       6   2013  106.6
23 2014-06-01       6   2014  110.8
24 2015-06-01       6   2015  114.9
25 2016-06-01       6   2016  116.7
26 2017-06-01       6   2017

In [87]:
# merge cpi for each year every quarter 2
adjusted_output = output.merge(cpi, 
                               how= 'left',
                               left_on= 'year',
                               right_on= 'year2')
adjusted_output

,code,ring_type,area,date,year_x,month,quarter,median_price,strata_title,year_y,month,year2,CPI
0,150,Inner Ring,Ashfield,1991-03-01,1991,3,1991Q1,137.0,all dwelling,1991-06-01,6,1991,57.2
1,150,Inner Ring,Ashfield,1991-06-01,1991,6,1991Q2,141.0,all dwelling,1991-06-01,6,1991,57.2
2,150,Inner Ring,Ashfield,1991-09-01,1991,9,1991Q3,142.0,all dwelling,1991-06-01,6,1991,57.2
3,150,Inner Ring,Ashfield,1991-12-01,1991,12,1991Q4,145.0,all dwelling,1991-06-01,6,1991,57.2
4,150,Inner Ring,Ashfield,1992-03-01,1992,3,1992Q1,158.0,all dwelling,1992-06-01,6,1992,54.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6190,0,GreaterSydney,Greater Sydney,2016-03-01,2016,3,2016Q1,760.0,all dwelling,2016-06-01,6,2016,116.7
6191,0,GreaterSydney,Greater Sydney,2016-06-01,2016,6,2016Q2,780.0,all dwelling,2016-06-01,6,2016,116.7
6192,0,GreaterSydney,Greater Sydney,2016-09-01,2016,9,2016Q3,788.0,all dwelling,2016-06-01,6,2016,116.7
6193,0,GreaterSydney,Greater Sydney,2016-12-01,2016,12,2016Q4,840.0,all dwelling,2016-06-01,6,2016,116.7


In [90]:
# assign CPI value for the rest
output_ns['CPI_quarter2'] = adjusted_output['CPI']
output_strata['CPI_quarter2'] = adjusted_output['CPI']

In [91]:
# rename easy-to-read header 
def rename(sheet):
    sheet = sheet.rename(columns={
    'year_x': 'year',
    'median_price': 'nominal_price',
    'CPI': 'CPI_quarter2'
    }
    )
    return sheet
adjusted_ad = rename(adjusted_output)
adjusted_ns = rename(output_ns)
adjusted_stra = rename(output_strata)

 `we set CPI quarter 2 in 2017 as base to adjust nominal price.`

In [96]:
# Adjusted price following the CPI in 2017 (set quarter 2 as base)
CPI_2017 = 120.6

# formula and round 2 decimal 
adjusted_ad['real_price'] = adjusted_ad['nominal_price'] * (CPI_2017/adjusted_ad["CPI_quarter2"]).round(2)

#adjusting the price in real (2017 Q2) dollars
adjusted_ad

,code,ring_type,area,date,year,month,quarter,nominal_price,strata_title,year_y,month,year2,CPI_quarter2,real_price
0,150,Inner Ring,Ashfield,1991-03-01,1991,3,1991Q1,137.0,all dwelling,1991-06-01,6,1991,57.2,289.07
1,150,Inner Ring,Ashfield,1991-06-01,1991,6,1991Q2,141.0,all dwelling,1991-06-01,6,1991,57.2,297.51
2,150,Inner Ring,Ashfield,1991-09-01,1991,9,1991Q3,142.0,all dwelling,1991-06-01,6,1991,57.2,299.62
3,150,Inner Ring,Ashfield,1991-12-01,1991,12,1991Q4,145.0,all dwelling,1991-06-01,6,1991,57.2,305.95
4,150,Inner Ring,Ashfield,1992-03-01,1992,3,1992Q1,158.0,all dwelling,1992-06-01,6,1992,54.0,352.34
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6190,0,GreaterSydney,Greater Sydney,2016-03-01,2016,3,2016Q1,760.0,all dwelling,2016-06-01,6,2016,116.7,782.80
6191,0,GreaterSydney,Greater Sydney,2016-06-01,2016,6,2016Q2,780.0,all dwelling,2016-06-01,6,2016,116.7,803.40
6192,0,GreaterSydney,Greater Sydney,2016-09-01,2016,9,2016Q3,788.0,all dwelling,2016-06-01,6,2016,116.7,811.64
6193,0,GreaterSydney,Greater Sydney,2016-12-01,2016,12,2016Q4,840.0,all dwelling,2016-06-01,6,2016,116.7,865.20


In [97]:
# function to adjust real price for 3 sheets
def adjusted_realprice(sheeti):
    CPI_2017 = 120.6
    sheeti['real_price'] = sheeti['nominal_price'] * (CPI_2017/sheeti["CPI_quarter2"]).round(2)
    return sheeti

adjusted_ns = adjusted_realprice(adjusted_ns)
adjusted_s = adjusted_realprice(adjusted_stra)

print(adjusted_ns)
print(adjusted_s)

      code       ring_type            area       date  year  month quarter  \
0      150      Inner Ring        Ashfield 1991-03-01  1991      3  1991Q1   
1      150      Inner Ring        Ashfield 1991-06-01  1991      6  1991Q2   
2      150      Inner Ring        Ashfield 1991-09-01  1991      9  1991Q3   
3      150      Inner Ring        Ashfield 1991-12-01  1991     12  1991Q4   
4      150      Inner Ring        Ashfield 1992-03-01  1992      3  1992Q1   
...    ...             ...             ...        ...   ...    ...     ...   
6190     0  Greater Sydney  Greater Sydney 2016-03-01  2016      3  2016Q1   
6191     0  Greater Sydney  Greater Sydney 2016-06-01  2016      6  2016Q2   
6192     0  Greater Sydney  Greater Sydney 2016-09-01  2016      9  2016Q3   
6193     0  Greater Sydney  Greater Sydney 2016-12-01  2016     12  2016Q4   
6194     0  Greater Sydney  Greater Sydney 2017-03-01  2017      3  2017Q1   

      nominal_price strata_title  CPI_quarter2  real_price  
0 

In [98]:
# save files as xlxs 
adjusted_ad.to_csv("all_dwelling.csv", index= False)
adjusted_ns.to_csv("non_strata.csv", index= False)
adjusted_s.to_csv("strata.csv", index= False)